# Introduction

> In this notebook we will work on joining all war event datasets on one dataset for better analysis
> Datasets:
> * food_insecurity_and_violent_conflict_gaza_dataset
> * gaza_diplacement_orders_ocha
> * gaza_diplacement_orders_gazamaps
> * casualties_daily
> * infrastructure_damaged
> * palestine_israel_conflict

##  Joining Strategy

To build a unified and meaningful dataset, we will extract and consolidate the most critical attributes from each source. The final dataset will include:
* `report_date` — the common temporal key across all datasets
* `latitude`, `longitude`, and `location` — spatial information to enable geospatial analysis
* `event_description` — textual summaries of reported incidents or developments
* `actors_involved` — entities such as those responsible (targeting) and those affected (targeted)
* Aggregated figures — including totals for **losses**, **casualties**, and **fatalities** per date and/or location

## Loading Data

In [1]:
import pandas as pd
import os
import datetime

In [2]:
os.chdir("C:/Guest/15-Career/MIT-Emrging Talent Program/ET6-CDSP-group-03-repo")

#### 1. Create Dictionary for all datasets in War Events Category

In [3]:
datasets_path = {
    "food_sec_df": "1_datasets/data/clean_datasets/fivc_gaza_october_cleaned.csv",
    "gaza_maps_displacement": "1_datasets/data/clean_datasets/gaza_displacement_cleaned.csv",
    "casualties_daily": "1_datasets/data/clean_datasets/casualties_cleaned_data.csv",
    "infrastructure_damaged": "1_datasets/data/clean_datasets/infrastructure_damaged_clean.csv",
    "palestine_israel_conflict": "1_datasets/data/clean_datasets/palestine_israel_war_cleaned_data.csv",
}
datasets = {
    "food_sec_df": None,
    "gaza_maps_displacement": None,
    "casualties_daily": None,
    "infrastructure_damaged": None,
    "palestine_israel_conflict": None,
}

In [49]:
for df_name, path in datasets_path.items():
    datasets[df_name] = pd.read_csv(path)

#### 2. Rename Date columns to be unified

In [50]:
datasets["food_sec_df"].rename({"Date": "report_date"}, axis=1, inplace=True)
datasets["gaza_maps_displacement"].rename({"date": "report_date"}, axis=1, inplace=True)
datasets["palestine_israel_conflict"].rename(
    {"date": "report_date"}, axis=1, inplace=True
)

In [51]:
for df_name, df in datasets.items():
    df["report_date"] = pd.to_datetime(df["report_date"])

In [52]:
full_date_range = pd.date_range(
    datetime.date(year=2023, month=10, day=1), datetime.date(year=2025, month=8, day=1)
)
full_dates = pd.DataFrame({"report_date": full_date_range})
full_dates["report_date"] = pd.to_datetime(full_dates["report_date"])

In [56]:
displacement_figures = (
    datasets["gaza_maps_displacement"]
    .groupby("report_date")
    .agg(
        displacement_orders_count=("id", "count"),
        avg_evacuated_area_km=("area_sq_km_displacement", "mean"),
    )
    .reset_index()
)

In [62]:
final_df = pd.merge(full_dates, displacement_figures, how="outer", on="report_date")

In [64]:
final_df.fillna(0, inplace=True)

#### 3. Joining Damaged Infrastucture Data

In [ ]:
infra_damaged_figures = (
    datasets["infrastructure_damaged"]
    .groupby("report_date")
    .agg(
        destroyed_edu=("edu_ext_destroyed", "sum"),
        damaged_edu=("edu_ext_damaged", "sum"),
        destroyed_civic_buildings=("civic_ext_destroyed", "sum"),
        destroyed_residential_buildings=("residential_ext_destroyed", "sum"),
    )
)
infra_damaged_figures

,destroyed_edu,damaged_edu,destroyed_civic_buildings,destroyed_residential_buildings
report_date,,,,
2023-10-07,1,15,5,80
2023-10-08,1,30,11,159
2023-10-09,2,45,16,790
2023-10-10,2,60,22,1009
2023-10-11,3,75,27,2835
...,...,...,...,...
2025-05-04,142,364,224,210000
2025-05-05,142,364,224,210000
2025-05-06,142,364,224,210000


In [76]:
final_df = pd.merge(final_df, infra_damaged_figures, how="outer", on="report_date")

Note! We have na in new columns: 

In [84]:
final_df.isna().sum()

report_date                         0
displacement_orders_count           0
avg_evacuated_area_km               0
destroyed_edu                      91
damaged_edu                        91
destroyed_civic_buildings          91
destroyed_residential_buildings    91
dtype: int64

#### 4. Joining Palestine war Events Data

In [91]:
datasets["palestine_israel_conflict"].replace(
    {
        "primary_targeting_actor": {
            "Military Forces of Israel (2022-)": "Military Forces of Israel"
        }
    },
    inplace=True,
)

In [ ]:
pic_df = datasets["palestine_israel_conflict"]
israeli_events_figures = pic_df[pic_df['primary_targeting_actor'] == "Military Forces of Israel"].groupby('report_date').agg(
  events_by_israeli_forces=("event_id", "count"),
)

pic_figures = (
    datasets["palestine_israel_conflict"]
    .groupby("report_date")
    .agg(
        total_num_events=("event_id", "count"),
      
    )
)
pic_figures = pd.merge(pic_figures, israeli_events_figures, how = 'outer', on = 'report_date')
pic_figures

,total_num_events,events_by_israeli_forces
report_date,,
2023-10-07,21,19
2023-10-08,31,31
2023-10-09,34,33
2023-10-10,26,26
2023-10-11,49,49
...,...,...
2025-06-16,37,30
2025-06-17,33,29
2025-06-18,26,23


In [104]:
final_df = pd.merge(final_df, pic_figures, how = 'outer', on = 'report_date')

In [106]:
final_df.head(10)

,report_date,displacement_orders_count,avg_evacuated_area_km,destroyed_edu,damaged_edu,destroyed_civic_buildings,destroyed_residential_buildings,total_num_events,events_by_israeli_forces
0,2023-10-01,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-10-02,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-10-03,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-10-04,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-10-05,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
5,2023-10-06,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
6,2023-10-07,1.0,27.0,1.0,15.0,5.0,80.0,21.0,19.0
7,2023-10-08,0.0,0.0,1.0,30.0,11.0,159.0,31.0,31.0
8,2023-10-09,0.0,0.0,2.0,45.0,16.0,790.0,34.0,33.0
9,2023-10-10,0.0,0.0,2.0,60.0,22.0,1009.0,26.0,26.0
